In [ ]:
%load ../lib/RecursiveSet.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

# Murder in the Family
---
We all know *Alice* and *Bob* from Cryptography.  However, little did
we now about their difficult family relationships, which might be the reason for
their secrecy.  

Unfortunately, things have escalated recently:
Murder occurred one evening in the home of their father and mother.  Alice and Bob are the only children of their parents. One member of the family *murdered* another member, the third member *witnessed* the crime, and the fourth member was an *accessory after the fact*.  Furthermore, we know the following:

1. *The accessory and the witness were of opposite sex*.
2. *The oldest member and the witness were of opposite sex*.
3. *The youngest member and the victim were of opposite sex*.
4. *The accessory was older than the victim*.
5. *The father was the oldest member*.
6. *The murderer was not the youngest member*.

**Question:** Which of the four—father, mother, Alice, or Bob—was the murderer?

We use the following variables:
* `FM`, `MM`, `AM`, `BM` are true if the father, the mother, Alice, or Bob is the murderer.
* `FW`, `MW`, `AW`, `BW` are true if the father, the mother, Alice, or Bob is the witness.
* `FV`, `MV`, `AV`, `BV` are true if the father, the mother, Alice, or Bob is the victim.
* `FA`, `MA`, `AA`, `BA` are true if the father, the mother, Alice, or Bob is the accessory.
* `AY` is true if Alice is younger than Bob.

In [ ]:
var P = set(
    "FM", "MM", "AM", "BM",
    "FW", "MW", "AW", "BW",
    "FV", "MV", "AV", "BV",
    "FA", "MA", "AA", "BA",
    "AY"
);
P

We first construct helper constraints to ensure each person has exactly one role, and each role is assigned to exactly one person.

The function `atLeastOne` takes a set of variables `V` and returns a propositional formula that is true if and only if at least one variable from the set `V` is true.  The static method `String.join` joins the elements of a collection of strings using the given separator.

In [ ]:
String atLeastOne(RecursiveSet<String> V) {
    return String.join(" ∨ ", V);
}

In [ ]:
atLeastOne(set("a", "b", "c"))

The function `atMostOne` takes a set of variables `V` and returns a propositional formula that is true if and only if at most one variable from the set `V` is true.

In [ ]:
String atMostOne(RecursiveSet<String> S) {
    var P = flatMap(S, p -> S.filterMap(q -> !p.equals(q),
                                        q -> "¬(" + p + " ∧ " + q + ")"));
    return String.join(" ∧ ", P);
}

In [ ]:
atMostOne(set("a", "b", "c"))

The function `exactlyOne` takes a set of variables `V` and returns a propositional formula that is true if and only if exactly one variable from the set `V` is true.

In [ ]:
String exactlyOne(RecursiveSet<String> V) {
    return "your code here";
}

In [ ]:
exactlyOne(set("a", "b", "c"))

In [ ]:
var Formulas = set(
    exactlyOne(set("FM", "FW", "FV", "FA")),
    exactlyOne(set("MM", "MW", "MV", "MA")),
    exactlyOne(set("AM", "AW", "AV", "AA")),
    exactlyOne(set("BM", "BW", "BV", "BA")),
    exactlyOne(set("FM", "MM", "AM", "BM")),
    exactlyOne(set("FW", "MW", "AW", "BW")),
    exactlyOne(set("FV", "MV", "AV", "BV")),
    exactlyOne(set("FA", "MA", "AA", "BA"))
);

**Fact 1**: The accessory and the witness were of opposite sex.

In [ ]:
var f1 = "your code here";

**Fact 2 & 5**: The oldest member and the witness were of opposite sex. The father was the oldest member.

In [ ]:
var f2 = "your code here";

**Fact 3**: The youngest member and the victim were of opposite sex.

In [ ]:
var f3a = "your code here";
var f3b = "your code here";

**Fact 4**: The accessory was older than the victim.
Father > Mother > Older Child > Younger Child.

In [ ]:
// your code here

**Fact 6**: The murderer was not the youngest member.

In [ ]:
// your code here

In [ ]:
Formulas = Formulas.union(set(f1, f2, f3a, f3b, "your code here"));

We need to transform the strings in the set `Formulas` into formulas, i.e. into trees of objects.  To this end we load the record classes representing formulas and the parser for propositional formulas.

In [ ]:
%load Formula.jsh
%load PropositionalLogicParser.jsh

In [ ]:
var Gs = Formulas.stream().map(f -> parse(f)).toList();

The function `evaluate(F, I)` takes a propositional formula $F$ and a propositional variable assignment $I$ and evaluates $F$ using the assignment $I$. We define it below to handle our logic operators.

In [ ]:
boolean evaluate(Formula f, RecursiveSet<String> I) {
    return switch (f) {
        case Var(var p)            -> I.contains(p);
        case Verum()               -> true;
        case Falsum()              -> false;
        case Not(var g)            -> !evaluate(g, I);
        case And(var g, var h)     -> evaluate(g, I) && evaluate(h, I);
        case Or(var g, var h)      -> evaluate(g, I) || evaluate(h, I);
        case Implies(var g, var h) -> !evaluate(g, I) || evaluate(h, I);
        case Equiv(var g, var h)   -> evaluate(g, I) == evaluate(h, I);
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

The function `allTrue(Fs, I)` takes a list of propositional formulas `Fs`
and a propositional variable assignment `I`.  It returns `true` only if all formulas from `Fs` are 
`true` given the variable assignment `I`.

In [ ]:
boolean allTrue(List<Formula> Gs, RecursiveSet<String> I) {
    return Gs.stream().allMatch(f -> evaluate(f, I));
}

We use the powerset function to filter all possible subsets of variables representing possible valid assignments:

In [ ]:
var validAssignments = P.powerset().filter(I -> allTrue(Gs, I));
validAssignments

Running this model limits the valid assignments to a single solution.